# Recipe Override with BenchMarkEvaluator

This notebook demonstrates the **recipe override** feature for model evaluation with `BenchMarkEvaluator`.

You can use a recipe file to define evaluation configuration (inference parameters, benchmark settings)
and selectively override specific values at runtime.

> **Note:** Actual execution requires valid AWS credentials and an appropriate IAM role. This notebook shows the pattern and API usage.

In [1]:
# Setup environment
import sys
import os
sys.path.insert(0, '../../sagemaker-train/src')
sys.path.insert(0, '../../sagemaker-core/src')

# Point botocore at the bundled service model
os.environ['AWS_DATA_PATH'] = os.path.abspath('../../sagemaker-core/sample')

!pip install -q omegaconf pyyaml

/Users/nargokul/.zshenv:1: command not found: 1805912728

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
pyenv: cannot rehash: /Users/nargokul/.pyenv/shims/.pyenv-shim exists


## Step 1: Create an Evaluation Recipe YAML File

The recipe defines inference parameters and evaluation settings in a reusable format.

In [2]:
import yaml
import os

# Define the evaluation recipe
eval_recipe_config = {
    "inference": {
        "max_new_tokens": 256,
        "temperature": 0.7,
        "top_p": 0.9,
        "do_sample": True
    },
    "evaluation": {
        "num_fewshot": 5,
        "batch_size": 16
    }
}

# Write the recipe to a local YAML file
recipe_path = "my_eval_recipe.yaml"
with open(recipe_path, "w") as f:
    yaml.dump(eval_recipe_config, f, default_flow_style=False)

print(f"Evaluation recipe written to: {os.path.abspath(recipe_path)}")
print("\nContents:")
with open(recipe_path) as f:
    print(f.read())

Evaluation recipe written to: /Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/my_eval_recipe.yaml

Contents:
evaluation:
  batch_size: 16
  num_fewshot: 5
inference:
  do_sample: true
  max_new_tokens: 256
  temperature: 0.7
  top_p: 0.9



## Step 2: Create BenchMarkEvaluator with Recipe and Overrides

We'll evaluate `amazon-nova-pro-v2` on the MMLU benchmark, using our recipe file
as the base config and overriding specific inference parameters.

In [3]:
from sagemaker.train.evaluate import BenchMarkEvaluator, get_benchmarks

Benchmark = get_benchmarks()

# Create evaluator with recipe + overrides
evaluator = BenchMarkEvaluator(
    benchmark=Benchmark.MMLU,
    subtasks=["abstract_algebra", "anatomy"],
    model="nova-textgeneration-lite-v2",
    s3_output_path="s3://sagemaker-us-west-2-211125564141/recipe-override-test/eval-output/",
    model_package_group="my-fine-tuned-models",
    # Recipe file for base evaluation config
    recipe="my_eval_recipe.yaml",
    # Override inference params for this run
    overrides={
        "inference": {
            "max_new_tokens": 512,
            "temperature": 0.1,
        }
    },
)

print("BenchMarkEvaluator configured with recipe + overrides.")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-core/src/sagemaker/core/shapes/shapes.py:7749: SyntaxWarning: invalid escape sequence '\|'
  """
/Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-core/src/sagemaker/core/shapes/shapes.py:8554: SyntaxWarning: invalid escape sequence '\*'
  """
/Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-core/src/sagemaker/core/shapes/shapes.py:8910: SyntaxWarning: invalid escape sequence '\*'
  """
/Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-

[06/05/26 11:12:22] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=910387;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=938674;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/nargokul/Library/Application Support/sagemaker/config.yaml


[06/05/26 11:12:23] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=270473;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=661574;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

                    INFO     Runs on sagemaker prod, region:us-west-2                                  ]8;id=939578;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-core/src/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=417811;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-core/src/sagemaker/core/utils/utils.py#370\370]8;;\

[06/05/26 11:12:25] INFO     Using first available ready MLflow app:                          ]8;id=672878;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=642890;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/common_utils/finetune_utils.py#137\137]8;;\
                             arn:aws:sagemaker:us-west-2:211125564141:mlflow-app/app-X4PV75IB                      
                             XLBR                                                                                  

                    INFO     Resolved MLflow resource ARN:                                    ]8;id=755369;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=941133;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/base_evaluator.py#177\177]8;;\
                             arn:aws:sagemaker:us-west-2:211125564141:mlflow-app/app-X4PV75IB                      
                             XLBR                                                                                  

                    INFO     Model package group provided as name: my-fine-tuned-models.      ]8;id=191474;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=665973;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/base_evaluator.py#214\214]8;;\
                             Fetching ModelPackageGroup object...                                                  

                    INFO     Resolved model package group name 'my-fine-tuned-models' to ARN: ]8;id=811746;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=986105;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/base_evaluator.py#233\233]8;;\
                             arn:aws:sagemaker:us-west-2:211125564141:model-package-group/my-                      
                             fine-tuned-models                                                                     

BenchMarkEvaluator configured with recipe + overrides.


## Step 3: Verify Configuration with `get_resolved_recipe()`

Before running the evaluation, inspect the resolved config to confirm that
overrides are applied correctly.

In [4]:
# Inspect the resolved configuration
resolved = evaluator.get_resolved_recipe()

print("Resolved evaluation recipe:")
print("=" * 40)
for section, params in resolved.items():
    print(f"\n[{section}]")
    if isinstance(params, dict):
        for key, value in params.items():
            print(f"  {key}: {value}")
    else:
        print(f"  {params}")

print("\n" + "=" * 40)
print("\nPrecedence verification:")
print(f"  max_new_tokens: {resolved['inference']['max_new_tokens']}")
print(f"    -> 512 from override (recipe had 256)")
print(f"  temperature: {resolved['inference']['temperature']}")
print(f"    -> 0.1 from override (recipe had 0.7)")
print(f"  top_p: {resolved['inference']['top_p']}")
print(f"    -> 0.9 from recipe (no override)")
print(f"  do_sample: {resolved['inference']['do_sample']}")
print(f"    -> True from recipe (no override)")

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=606541;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-core/src/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=88972;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-core/src/sagemaker/core/telemetry/telemetry_logging.py#103\103]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     Fetching evaluation override parameters for hyperparameters ]8;id=63879;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/benchmark_evaluator.py\benchmark_evaluator.py]8;;\:]8;id=461610;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/benchmark_evaluator.py#477\477]8;;\
                             property                                                                              

                    INFO     Fetching hub content metadata for nova-textgeneration-lite-v2 from ]8;id=644419;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/common_utils/recipe_utils.py\recipe_utils.py]8;;\:]8;id=177816;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/common_utils/recipe_utils.py#201\201]8;;\
                             SageMakerPublicHub                                                                    

                    INFO     Searching for evaluation recipe with Type='Evaluation' and         ]8;id=884564;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/common_utils/recipe_utils.py\recipe_utils.py]8;;\:]8;id=669710;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/common_utils/recipe_utils.py#221\221]8;;\
                             EvaluationType='DeterministicTextBenchmark'                                           

                    INFO     Downloading override parameters from                               ]8;id=376096;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/common_utils/recipe_utils.py\recipe_utils.py]8;;\:]8;id=818341;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/common_utils/recipe_utils.py#246\246]8;;\
                             s3://jumpstart-cache-prod-us-west-2/recipes/nova_lite_2_0_p5_48xl_                    
                             gpu_general_text_benchmark_eval_override_params_sm_jobs_v2.2.0.jso                    
                             n                                                                                     

...

[06/05/26 11:12:27] WARNING  Unknown key 'evaluation' in user recipe will be included but is ]8;id=374535;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/recipe_resolver.py\recipe_resolver.py]8;;\:]8;id=491738;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/recipe_resolver.py#338\338]8;;\
                             not part of the base template. It will not be validated.                              

                    WARNING  Unknown key 'evaluation.batch_size' in user recipe will be      ]8;id=141933;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/recipe_resolver.py\recipe_resolver.py]8;;\:]8;id=120816;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/recipe_resolver.py#338\338]8;;\
                             included but is not part of the base template. It will not be                         
                             validated.                                                                            

                    WARNING  Unknown key 'evaluation.num_fewshot' in user recipe will be     ]8;id=810039;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/recipe_resolver.py\recipe_resolver.py]8;;\:]8;id=328882;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/recipe_resolver.py#338\338]8;;\
                             included but is not part of the base template. It will not be                         
                             validated.                                                                            

                    WARNING  Unknown key 'inference.do_sample' in user recipe will be        ]8;id=240274;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/recipe_resolver.py\recipe_resolver.py]8;;\:]8;id=407656;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/recipe_resolver.py#338\338]8;;\
                             included but is not part of the base template. It will not be                         
                             validated.                                                                            

Resolved evaluation recipe:

[inference]
  max_new_tokens: 512
  temperature: 0.1
  top_k: -1
  top_p: 0.9
  do_sample: True

[evaluation]
  batch_size: 16
  num_fewshot: 5


Precedence verification:
  max_new_tokens: 512
    -> 512 from override (recipe had 256)
  temperature: 0.1
    -> 0.1 from override (recipe had 0.7)
  top_p: 0.9
    -> 0.9 from recipe (no override)
  do_sample: True
    -> True from recipe (no override)


## Step 4: Validate Before Running

A good practice is to programmatically verify the resolved config matches expectations
before launching a potentially expensive evaluation job.

In [5]:
# Programmatic validation
resolved = evaluator.get_resolved_recipe()

inference_config = resolved["inference"]
eval_config = resolved["evaluation"]

# Verify overrides took effect
assert inference_config["max_new_tokens"] == 512, "Override should set max_new_tokens to 512"
assert inference_config["temperature"] == 0.1, "Override should set temperature to 0.1"

# Verify recipe file values are preserved where not overridden
assert inference_config["top_p"] == 0.9, "Recipe value should be preserved"
assert inference_config["do_sample"] == True, "Recipe value should be preserved"
assert eval_config["num_fewshot"] == 5, "Recipe value should be preserved"
assert eval_config["batch_size"] == 16, "Recipe value should be preserved"

print("All validation checks passed!")
print("\nFinal inference config for evaluation:")
print(f"  max_new_tokens: {inference_config['max_new_tokens']} (overridden)")
print(f"  temperature:    {inference_config['temperature']} (overridden)")
print(f"  top_p:          {inference_config['top_p']} (from recipe)")
print(f"  do_sample:      {inference_config['do_sample']} (from recipe)")
print(f"\nEvaluation config:")
print(f"  num_fewshot:    {eval_config['num_fewshot']} (from recipe)")
print(f"  batch_size:     {eval_config['batch_size']} (from recipe)")

All validation checks passed!

Final inference config for evaluation:
  max_new_tokens: 512 (overridden)
  temperature:    0.1 (overridden)
  top_p:          0.9 (from recipe)
  do_sample:      True (from recipe)

Evaluation config:
  num_fewshot:    5 (from recipe)
  batch_size:     16 (from recipe)


## Step 5: Run Evaluation (Requires AWS Credentials)

When satisfied with the configuration, launch the evaluation job.

In [6]:
# Run evaluation
execution = evaluator.evaluate()
print(f"Evaluation started: {execution}")

# Wait for completion
# execution.wait()

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=300284;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-core/src/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=570968;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-core/src/sagemaker/core/telemetry/telemetry_logging.py#103\103]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     Getting or creating artifact for source:                         ]8;id=21207;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=975004;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/base_evaluator.py#680\680]8;;\
                             arn:aws:sagemaker:us-west-2:aws:hub-content/SageMakerPublicHub/M                      
                             odel/nova-textgeneration-lite-v2/3.50.0                                               

                    INFO     Searching for existing artifact for base model:                  ]8;id=392138;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=253953;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/base_evaluator.py#537\537]8;;\
                             arn:aws:sagemaker:us-west-2:aws:hub-content/SageMakerPublicHub/M                      
                             odel/nova-textgeneration-lite-v2/3.50.0                                               

                    INFO     Found existing artifact:                                         ]8;id=831525;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=858543;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/base_evaluator.py#546\546]8;;\
                             arn:aws:sagemaker:us-west-2:211125564141:artifact/2350727a8e9da2                      
                             39420b04b766dc45d3                                                                    

                    INFO     Using user-provided model_package_group ARN:                     ]8;id=150140;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=149050;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/base_evaluator.py#490\490]8;;\
                             arn:aws:sagemaker:us-west-2:211125564141:model-package-group/my-                      
                             fine-tuned-models                                                                     

...

                    INFO     Final Resource Status: Active                                       ]8;id=384890;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=329510;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-core/src/sagemaker/core/resources.py#26828\26828]8;;\

                    INFO     Pipeline                                                              ]8;id=584482;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=98352;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/execution.py#119\119]8;;\
                             SagemakerEvaluation-BenchmarkEvaluation-668611b1-7c0e-4f9d-bc7d-34019                 
                             3190e91 is now active and ready for execution                                         

                    INFO     Starting pipeline execution: eval-nova-1b752f2c-1780683149            ]8;id=360350;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=123028;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/execution.py#309\309]8;;\

                    INFO     Pipeline execution started:                                           ]8;id=478289;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=347936;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/evaluate/execution.py#320\320]8;;\
                             arn:aws:sagemaker:us-west-2:211125564141:pipeline/SagemakerEvaluation                 
                             -BenchmarkEvaluation-668611b1-7c0e-4f9d-bc7d-340193190e91/execution/q                 
                             8ygljaackqx                                                                           

Evaluation started: arn='arn:aws:sagemaker:us-west-2:211125564141:pipeline/SagemakerEvaluation-BenchmarkEvaluation-668611b1-7c0e-4f9d-bc7d-340193190e91/execution/q8ygljaackqx' name='eval-nova-1b752f2c' status=PipelineExecutionStatus(overall_status='Executing', step_details=[], failure_reason=None) last_modified_time=datetime.datetime(2026, 6, 5, 11, 12, 29, 547000, tzinfo=tzlocal()) eval_type=<EvalType.BENCHMARK: 'benchmark'> s3_output_path='s3://sagemaker-us-west-2-211125564141/recipe-override-test/eval-output/' steps=[]


## Summary

Using recipe overrides with `BenchMarkEvaluator`:

- **Define once** — Base inference and evaluation config in a YAML recipe file
- **Override at runtime** — Adjust parameters for specific eval runs without editing the file
- **Verify before running** — `get_resolved_recipe()` shows the exact config that will be used
- **Reproducibility** — Recipe files can be version-controlled and shared across the team

**Common override patterns for evaluation:**
- Lower `temperature` (e.g., 0.1) for more deterministic benchmark scoring
- Increase `max_new_tokens` for benchmarks requiring longer outputs
- Adjust `num_fewshot` to test different few-shot settings